# VQ-VAE-2 Latents — Codebook Analysis

Codebook usage histograms, style-codebook perplexity, style-path utilisation, dominant pairs, MI/χ² discriminativeness, and histogram PCA/t-SNE.

In [ ]:
import sys
import os

# Add project root to sys.path (this notebook lives in eval/)
sys.path.insert(0, os.path.abspath(".."))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import json

# Import local modules
import models.vqvae as vqvae
import utils.utils as utils
from utils.utils import load_data, CreateBrainMaskd, ApplyBrainMaskd

# ============================================================
# CONFIGURATION
# ============================================================
# Use vqvae_best.pt for the best checkpoint (by rolling avg total loss),
# or vqvae_model.pt for the latest checkpoint.
CHECKPOINT_PATH = (
    "/home/ng24/projects/multiview-crl/results/ADNI_registered/multiview-06-content-lr-002-single-level/vqvae_best1.pt"
)
DATA_DIR = "/data/natalia/ADNI_stripped"
CSV_PATH = "/home/ng24/projects/nmpevqvae/labels_cleaned_3class.csv"
NUM_SAMPLES = 200  # number of subjects to process
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ADNI 2-view split (must match training)
CONTENT_SIZE = 256  # dimensions treated as content (shared between T1/T2)
STYLE_SIZE = 256  # dimensions treated as style (view-specific)

# Resampling: "original" (1 mm, ~182x218x182) or "2mm" (91x109x91)
RESAMPLE_MODE = "original"
# ============================================================

print(f"Using device: {DEVICE}")

## 1. Load checkpoint & build model

In [ ]:
# Load settings saved alongside the checkpoint
settings_path = os.path.join(os.path.dirname(CHECKPOINT_PATH), "settings.json")
with open(settings_path, "r") as f:
    settings = json.load(f)

print("Model settings:")
for k in [
    "vqvae_hidden_channels",
    "vqvae_res_channels",
    "vqvae_nb_res_layers",
    "vqvae_nb_levels",
    "vqvae_embed_dim",
    "vqvae_nb_entries",
    "vqvae_scaling_rates",
]:
    print(f"  {k}: {settings.get(k, 'N/A')}")

# Override CONTENT_SIZE / STYLE_SIZE from saved settings.
# settings.json stores content_indices (list[list[int]]) and style_indices (list[int])
# written by update_args() at training time — use them so the VQVAE is rebuilt
# with exactly the same content/style ratio that was used during training.
# Falling back to the hardcoded values only when the keys are absent (old checkpoints).
if "content_indices" in settings and "style_indices" in settings:
    CONTENT_SIZE = len(settings["content_indices"][0])
    STYLE_SIZE = len(settings["style_indices"])
    print(f"\n  content_size : {CONTENT_SIZE}  (from settings.json['content_indices'])")
    print(f"  style_size   : {STYLE_SIZE}  (from settings.json['style_indices'])")
else:
    print(f"\n  ⚠ content_indices / style_indices not found in settings.json.")
    print(f"    Using hardcoded CONTENT_SIZE={CONTENT_SIZE}, STYLE_SIZE={STYLE_SIZE}.")
    print(f"    If these don't match the training run, content_channels will be wrong.")

In [ ]:
# Build model — must match the exact architecture used during training.
# inject_style_to_decoder controls whether decoders[0] has a final_conv branch;
# if it differs from the checkpoint the decoder weights will silently not load.
inject_style = settings.get("inject_style_to_decoder", False)

# ── Auto-detect content/style split from checkpoint weights ──────────────────
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
state_dict = checkpoint["encoders"]

# Strip MoCo "online." prefix if present to find the right keys
_prefix = ""
if any(k.startswith("online.") for k in state_dict):
    _prefix = "online."

# channel_logits is now a ParameterDict keyed by level.
# Old checkpoints: "module.channel_logits" (single param)
# New checkpoints: "module.channel_logits.0", "module.channel_logits.1", etc.
_cl_key_old = f"{_prefix}module.channel_logits"

# Find all channel_logits.X keys where X is a digit
_cl_levels = []
for k in state_dict:
    stripped = k[len(_prefix) :] if k.startswith(_prefix) else k
    if stripped.startswith("module.channel_logits."):
        suffix = stripped[len("module.channel_logits.") :]
        if suffix.isdigit():
            _cl_levels.append(int(suffix))
_cl_levels = sorted(_cl_levels)

# ── Detect mask_mode from checkpoint keys ────────────────────────────────────
# fixed mode: has fixed_mask_* buffers, no channel_logits
# learned mode: has channel_logits params
# onthefly mode: neither (logits computed from activations at runtime)
_has_fixed_mask = any(
    (k[len(_prefix) :] if k.startswith(_prefix) else k).startswith("module.fixed_mask_") for k in state_dict
)

if _has_fixed_mask:
    _detected_mask_mode = "fixed"
elif _cl_levels or _cl_key_old in state_dict:
    _detected_mask_mode = "learned"
else:
    _detected_mask_mode = "onthefly"

# Allow settings.json to override (it's always authoritative if present)
_mask_mode = settings.get("mask_mode", _detected_mask_mode)

if _cl_levels:
    # New-style per-level channel_logits
    _cl_key_0 = f"{_prefix}module.channel_logits.{_cl_levels[0]}"
    _mask_dim = state_dict[_cl_key_0].shape[0]
    content_style_levels = _cl_levels
elif _cl_key_old in state_dict:
    # Old-style single channel_logits
    _mask_dim = state_dict[_cl_key_old].shape[0]
    content_style_levels = [0]
else:
    _mask_dim = None
    # No channel_logits (onthefly or fixed mode) — read content_style_levels from settings
    content_style_levels = settings.get("content_style_levels", [0])

# ── Per-level content_channels detection from codebook input sizes ────────────
hidden_channels = settings["vqvae_hidden_channels"]
_embed_dim = settings["vqvae_embed_dim"]
_nb_levels = settings["vqvae_nb_levels"]

# Detect per-level content_channels from codebook conv_in dimensions.
# Works for any mask_mode (learned, onthefly, or fixed) as long as the codebook was
# sized for the content subset (content_channels < hidden_channels).
_content_ch_per_level = {}
_used_codebook_detection = False
content_ratios = None

for lvl in content_style_levels:
    _cb_key = f"{_prefix}module.codebooks.{lvl}.conv_in.weight"
    if _cb_key in state_dict:
        _cb_in = state_dict[_cb_key].shape[1]
        if lvl == _nb_levels - 1:
            # Top level: codebook input = content_channels (no embed_dim concat)
            _content_ch_per_level[lvl] = _cb_in
        else:
            # Lower levels: codebook input = content_channels + embed_dim
            _content_ch_per_level[lvl] = _cb_in - _embed_dim

# Check if codebook detection is valid: if ALL detected levels give
# content_channels == hidden_channels, the checkpoint predates per-level
# codebook sizing — fall back to settings-based CONTENT_SIZE / STYLE_SIZE.
_all_full_width = all(cc == hidden_channels for cc in _content_ch_per_level.values()) if _content_ch_per_level else True

if not _all_full_width:
    # Codebook was sized for content masking — use detected values
    _used_codebook_detection = True

    _first_lvl = content_style_levels[0]
    _content_channels = _content_ch_per_level.get(_first_lvl, hidden_channels)
    _style_channels = hidden_channels - _content_channels

    CONTENT_SIZE = _content_channels
    STYLE_SIZE = _style_channels

    # Compute per-level content_ratios if levels have different content sizes
    _ratios = [_content_ch_per_level[lvl] / hidden_channels for lvl in content_style_levels]
    if len(set(_ratios)) > 1:
        content_ratios = _ratios

    _mode_desc = {
        "learned": "learned",
        "onthefly": "onthefly (no channel_logits)",
        "fixed": "fixed (first K = content)",
    }
    print(f"Auto-detected from checkpoint (per-level codebook sizing):")
    print(f"  mask_mode             : {_mode_desc.get(_mask_mode, _mask_mode)}")
    print(f"  content_style_levels  : {content_style_levels}")
    for lvl in content_style_levels:
        cc = _content_ch_per_level.get(lvl, "?")
        print(f"  level {lvl} content_ch    : {cc} / {hidden_channels}  ({cc/hidden_channels:.1%})")
    print(f"  -> CONTENT_SIZE={CONTENT_SIZE}, STYLE_SIZE={STYLE_SIZE}")
    if content_ratios:
        print(f"  -> content_ratios={content_ratios}")
else:
    # Old checkpoint: codebooks use full hidden_channels.
    # Keep CONTENT_SIZE / STYLE_SIZE from settings cell (cell above).
    print(f"Codebook detection gives content_ch == hidden_channels at all levels.")
    print(f"  content_style_levels  : {content_style_levels}")
    print(f"  → Using CONTENT_SIZE={CONTENT_SIZE}, STYLE_SIZE={STYLE_SIZE} from settings.json")

# ── Detect whether checkpoint uses hidden_channels or embed_dim masking ──────
if _mask_dim is not None and _mask_dim != hidden_channels:
    raise RuntimeError(
        f"Checkpoint has channel_logits of size {_mask_dim} but hidden_channels={hidden_channels}. "
        f"This checkpoint was likely trained with an older code version where the mask was on "
        f"embed_dim={_embed_dim}. You need to checkout the matching code version "
        f"to load this checkpoint."
    )

# ── Detect separate_encoders from checkpoint keys ────────────────────────────
# If the checkpoint contains "module.encoders_v1.*" keys, the model was trained
# with --separate-encoders.
_sep_enc = any(
    (k[len(_prefix) :] if k.startswith(_prefix) else k).startswith("module.encoders_v1.") for k in state_dict
)
if _sep_enc:
    print("Detected separate_encoders in checkpoint (encoders_v1 keys found).")

_style_inj_mode = settings.get("style_injection_mode", "concat")

# ── Read training flags that affect the encoder forward pass ─────────────────
# These MUST match training exactly, otherwise the encoder produces different
# features at inference time (e.g. style channels zeroed between levels when
# they should be passed through, causing out-of-distribution inputs at higher levels).
_pass_full = settings.get("pass_full_to_next_level", False)
_narrow_enc = settings.get("narrow_encoder_input", False)
_top_recon = settings.get("top_level_recon_only", False)
_quantize_style = settings.get("quantize_style", False)
_style_embed_dim = settings.get("style_embed_dim", None)
_style_nb_entries_raw = settings.get("style_nb_entries", None)
_use_content_proj = settings.get("use_content_projection", False)

# ── Normalize per-level codebook sizes ───────────────────────────────────────
# --vqvae-nb-entries / --style-nb-entries are argparse nargs="+" → always a
# list in settings.json. W&B sweeps occasionally round-trip these as
# stringified lists ("[384]") or nested lists ([[384]]), which then reach
# CodeLayer where torch.randn(embed_dim, nb_entries) raises
#   TypeError: randn(): argument 'size' failed to unpack ... got list
# Flatten + coerce to int / flat list[int] before handing to VQVAE.
import ast as _ast


def _normalize_entries(v):
    if v is None:
        return None
    if isinstance(v, str):
        try:
            v = _ast.literal_eval(v)
        except (ValueError, SyntaxError):
            return int(v)
    if isinstance(v, (list, tuple)):
        flat = []
        for x in v:
            if isinstance(x, (list, tuple)):
                flat.extend(x)
            else:
                flat.append(x)
        flat = [int(x) for x in flat]
        return flat[0] if len(flat) == 1 else flat
    return int(v)


_nb_entries = _normalize_entries(settings["vqvae_nb_entries"])
_style_nb_entries = _normalize_entries(_style_nb_entries_raw)
print(f"vqvae_nb_entries (normalized) : {_nb_entries}")
if _style_nb_entries is not None:
    print(f"style_nb_entries (normalized) : {_style_nb_entries}")

# ── Optional: skip levels in the final reconstruction decoder ────────────────
# Define SKIP_RECON_LEVELS in a cell above to ablate finer codes, e.g.
#     SKIP_RECON_LEVELS = [0, 1]
# zeros out levels 0 and 1 in the final (level-0) decoder concat, leaving only
# the top (coarsest) codes to drive reconstruction. The top level itself cannot
# be skipped. If undefined, falls back to whatever was stored at training time.
try:
    _skip_recon = SKIP_RECON_LEVELS  # noqa: F821 — optional user override
    print(f"⚠ Notebook override: skip_decoder_concat_levels={_skip_recon}")
except NameError:
    _skip_recon = settings.get("skip_decoder_concat_levels", None)
    if _skip_recon:
        print(f"skip_decoder_concat_levels (from settings): {_skip_recon}")

vqvae_model = vqvae.VQVAE(
    in_channels=1,
    hidden_channels=hidden_channels,
    res_channels=settings["vqvae_res_channels"],
    nb_res_layers=settings.get("vqvae_nb_res_layers", 2),
    nb_levels=_nb_levels,
    embed_dim=_embed_dim,
    nb_entries=_nb_entries,
    scaling_rates=settings["vqvae_scaling_rates"],
    content_size=CONTENT_SIZE,
    style_size=STYLE_SIZE,
    inject_style_to_decoder=inject_style,
    content_style_levels=content_style_levels,
    content_ratios=content_ratios,
    separate_encoders=_sep_enc,
    mask_mode=_mask_mode,
    style_injection_mode=_style_inj_mode,
    pass_full_to_next_level=_pass_full,
    narrow_encoder_input=_narrow_enc,
    top_level_recon_only=_top_recon,
    quantize_style=_quantize_style,
    style_embed_dim=_style_embed_dim,
    style_nb_entries=_style_nb_entries,
    use_content_projection=_use_content_proj,
    skip_decoder_concat_levels=_skip_recon,
)

content_channels = vqvae_model.content_channels
print(f"\nhidden_channels         : {hidden_channels}")
print(f"content_channels        : {content_channels}")
if vqvae_model.content_channels_per_level:
    print(f"content_channels_per_lvl: {vqvae_model.content_channels_per_level}")
print(f"content_style_levels    : {content_style_levels}")
print(f"mask_mode               : {_mask_mode}")
print(f"inject_style_to_decoder : {inject_style}")
print(f"separate_encoders       : {_sep_enc}")
print(f"style_injection_mode    : {_style_inj_mode}")
print(f"pass_full_to_next_level : {_pass_full}")
print(f"narrow_encoder_input    : {_narrow_enc}")
print(f"top_level_recon_only    : {_top_recon}")
print(f"quantize_style          : {_quantize_style}")
print(f"use_content_projection  : {_use_content_proj}")

# Wrap in DataParallel to match the saved checkpoint structure
vqvae_model = torch.nn.DataParallel(vqvae_model, device_ids=[0])

# ── Key remapping ─────────────────────────────────────────────────────────────
new_state_dict = {}
is_moco_checkpoint = any(k.startswith("online.") for k in state_dict)
if is_moco_checkpoint:
    print("Detected MoCoEncoder checkpoint — stripping 'online.' prefix.")

for k, v in state_dict.items():
    k = k.replace(".blocks.", ".stack.")  # legacy rename
    # Migrate old single channel_logits → ParameterDict key "0"
    if k.endswith("module.channel_logits") and not any(c.isdigit() for c in k.split(".")[-1]):
        k = k + ".0"
    if k.startswith("online."):
        new_state_dict[k[len("online.") :]] = v
    elif k.startswith(
        ("momentum_encoders.", "momentum_codebook_projs.", "momentum_content_proj.", "momentum_level0_proj.", "queue_")
    ):
        pass  # MoCo-only state — discard
    elif k.startswith("momentum_encoders_v1."):
        pass  # MoCo momentum copy of view-1 encoder — discard
    else:
        new_state_dict[k] = v

# ── Drop keys with shape mismatches (e.g. projection head from older config) ─
_model_sd = vqvae_model.state_dict()
_shape_skipped = []
for k in list(new_state_dict):
    if k in _model_sd and new_state_dict[k].shape != _model_sd[k].shape:
        _shape_skipped.append(f"{k}: ckpt {new_state_dict[k].shape} vs model {_model_sd[k].shape}")
        del new_state_dict[k]
if _shape_skipped:
    print(f"\u26a0 Skipped {len(_shape_skipped)} keys with shape mismatch:")
    for s in _shape_skipped:
        print(f"    {s}")

missing, unexpected = vqvae_model.load_state_dict(new_state_dict, strict=False)

missing_real = [k for k in missing if "num_batches_tracked" not in k]
unexpected_real = [k for k in unexpected if "num_batches_tracked" not in k]

if missing_real:
    print(f"\u26a0 Missing keys ({len(missing_real)}): {missing_real[:6]}{'...' if len(missing_real) > 6 else ''}")
if unexpected_real:
    print(
        f"\u26a0 Unexpected keys ({len(unexpected_real)}): {unexpected_real[:6]}{'...' if len(unexpected_real) > 6 else ''}"
    )
if not missing_real and not unexpected_real:
    print("\u2713 Checkpoint loaded cleanly")
elif not missing_real:
    print("\u2713 Checkpoint loaded (unexpected keys are harmless extras in the file)")

vqvae_model.to(DEVICE)
vqvae_model.eval()
print(f"\u2713 Model on {DEVICE}, step {checkpoint['step']}")
print(f"  Total params: {sum(p.numel() for p in vqvae_model.parameters()):,}")

In [ ]:
# -- Diagnostic: isolate where the constant-output problem lives --
import torch

vqvae_module = vqvae_model.module if hasattr(vqvae_model, "module") else vqvae_model

# 1. Check encoder weight statistics (are weights non-trivial?)
print("=== Encoder weight check ===")
for name, p in list(vqvae_module.encoders[0].named_parameters())[:6]:
    print(
        f"  {name}: shape={tuple(p.shape)}  "
        f"mean={p.data.mean():.6f}  std={p.data.std():.6f}  "
        f"absmax={p.data.abs().max():.6f}"
    )

# 2. Check ReZero alpha values (if 0, residual blocks are identity)
print("\n=== ReZero alpha values ===")
for name, p in vqvae_module.encoders[0].named_parameters():
    if "alpha" in name:
        print(f"  {name}: {p.data.item():.6f}")

# 3. Test raw encoder (bypass VQVAE forward)
print("\n=== Raw encoder test ===")
with torch.no_grad():
    d1 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)
    d2 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)

    # Run through encoder level 0 directly
    out1 = vqvae_module.encoders[0](d1)
    out2 = vqvae_module.encoders[0](d2)

    print(f"  Encoder output shape: {out1.shape}")
    print(f"  out1 range: [{out1.min():.4f}, {out1.max():.4f}]  mean={out1.mean():.6f}")
    print(f"  out2 range: [{out2.min():.4f}, {out2.max():.4f}]  mean={out2.mean():.6f}")

    # Check spatial variation (different voxels should have different values)
    print(f"  out1 spatial std (per-channel): {out1[0].std(dim=[1, 2, 3]).mean():.6f}")

    # Pool and compare
    p1 = out1.mean(dim=[2, 3, 4])
    p2 = out2.mean(dim=[2, 3, 4])
    cos = torch.nn.functional.cosine_similarity(p1, p2).item()
    print(f"  Pooled cosine similarity: {cos:.6f}")
    if cos > 0.999:
        print("  WARNING: Raw encoder also produces constant output!")
    else:
        print("  OK: Raw encoder works - bug is in VQVAE.forward()")

# 4. Test layer by layer to find where signal dies
print("\n=== Layer-by-layer signal propagation ===")
with torch.no_grad():
    x1 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)
    x2 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)
    for layer_idx, layer in enumerate(vqvae_module.encoders[0].layers):
        x1 = layer(x1)
        x2 = layer(x2)
        cos = torch.nn.functional.cosine_similarity(x1.flatten(1), x2.flatten(1)).item()
        print(
            f"  After layer {layer_idx} ({type(layer).__name__}): "
            f"shape={tuple(x1.shape)}  cos={cos:.6f}  "
            f"range=[{x1.min():.3f},{x1.max():.3f}]"
        )

## 2. Load dataset & transforms

In [ ]:
df = pd.read_csv(CSV_PATH)
label_values = sorted(df["Group"].unique())
label_map = {v: i for i, v in enumerate(label_values)}
label_names = {i: v for v, i in label_map.items()}
print(f"Labels: {label_map}")

items, missing = load_data(df, DATA_DIR, label_map)
print(f"Loaded {len(items)} samples, {len(missing)} missing files")
items = items[:NUM_SAMPLES]
print(f"Using first {len(items)} subjects")

In [ ]:
# Match the training val pipeline EXACTLY. Previously the notebook always
# applied CreateBrainMaskd (intensity > 0 threshold). Training with masks on
# disk skips that and loads the proper brain-mask .nii.gz directly via
# LoadImaged — a different mask → different NormalizeIntensityd stats →
# different features → different modality probe result. Fix: delegate to
# utils.utils.transforms and use masks_from_disk when the CSV items carry
# mask paths.
from utils.utils import transforms as get_transforms

# Read spacing and spatial_size from settings.json (saved at training time)
spacing = settings.get("image_spacing", 2.0)
crop_margin = settings.get("crop_margin", 0)

_saved_spatial = settings.get("spatial_size", None)
if _saved_spatial is not None:
    spatial_size = tuple(_saved_spatial)
elif spacing == 1.0:
    spatial_size = (182, 218, 182)
else:
    spatial_size = (91, 109, 91)

# Mirror data/datasets.py:1283 — masks_from_disk is True iff every loaded
# item carries both mask paths.
masks_from_disk = all(("mask_image" in it) and ("mask_z_image" in it) for it in items)

_, _val_transforms_raw = get_transforms(
    spacing=spacing,
    crop_margin=crop_margin,
    spatial_size=spatial_size,
    masks_from_disk=masks_from_disk,
    asymmetric_aug=False,
)

# The raw val transform expects:
#   - ``mask_t1``/``mask_t2`` paths when ``masks_from_disk=True`` (loaded by LoadImaged)
#   - a ``label`` key at the end (required by the final ToTensord in utils.utils.transforms)
# Existing notebook cells pass only ``image_t1``/``image_t2``. Wrap the
# transform to auto-inject mask paths and a placeholder label, keyed on the
# T1 image path.
_img_to_extras = {
    it["image"]: {
        "mask_t1": it.get("mask_image"),
        "mask_t2": it.get("mask_z_image"),
        "label": it.get("label", 0),
    }
    for it in items
}


def val_transforms(data_dict):
    d = dict(data_dict)
    extras = _img_to_extras.get(d.get("image_t1"), {})
    if masks_from_disk:
        if "mask_t1" not in d and extras.get("mask_t1") is not None:
            d["mask_t1"] = extras["mask_t1"]
        if "mask_t2" not in d and extras.get("mask_t2") is not None:
            d["mask_t2"] = extras["mask_t2"]
    if "label" not in d:
        d["label"] = extras.get("label", 0)
    return _val_transforms_raw(d)


print(f"Transforms ready — spacing={spacing}mm, spatial_size={spatial_size}, " f"masks_from_disk={masks_from_disk}")
if not masks_from_disk:
    print(
        "  (No disk masks found on items — using intensity-threshold CreateBrainMaskd.\n"
        "  If training ran with disk masks, the modality probe will differ from the training log.)"
    )

## 3. Feature extraction

We call `vqvae_model(img, pool_only=True, return_recon=False)` which returns:
- `encoder_features`: list of `(B, embed_dim)` pooled vectors, one per level
  - All levels are pooled from the codebook projection (embed_dim space)
  - When `channel_logits` is active, content/style split is in this embed_dim space
- `estimated_content_indices`: the Gumbel-selected embedding dim indices at level 0

For visualisation we split level-0 pooled features into **content** and **style** subsets
using `estimated_content_indices`.

In [ ]:
nb_levels = settings["vqvae_nb_levels"]

# Storage
all_features = {f"level_{i}": [] for i in range(nb_levels)}
all_labels = []
all_modalities = []
all_subjects = []
all_content_indices = {}  # dict mapping level -> view-0 (T1) content indices
all_content_indices_v1 = {}  # dict mapping level -> view-1 (T2) content indices (only set for per-view masks)

print(f"Extracting latent features from {len(items)} subjects (T1 + T2 each) ...")

vqvae_module = vqvae_model.module if hasattr(vqvae_model, "module") else vqvae_model

# Map modality name → view_idx for separate-encoder models.
# view 0 = T1 (first modality), view 1 = T2 (second modality).
_VIEW_IDX = {"T1": 0, "T2": 1}

# Detect per-view masks: separate encoders with channel_logits_v1
_has_per_view = (
    getattr(vqvae_module, "separate_encoders", False) and getattr(vqvae_module, "channel_logits_v1", None) is not None
)

for idx, item in enumerate(items):
    if idx % 20 == 0:
        print(f"  {idx}/{len(items)} ...")

    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for modality, key in [("T1", "image_t1"), ("T2", "image_t2")]:
        img = transformed[key].unsqueeze(0).to(DEVICE)  # (1, 1, D, H, W)

        with torch.no_grad():
            # Get soft_content_masks (7th returned element)
            _, _, enc_features, _, _, _, soft_masks, *_ = vqvae_model(
                img,
                return_recon=False,
                pool_only=True,
                view_idx=_VIEW_IDX.get(modality, 0),
            )

        # enc_features is a list of (1, hidden_channels) pooled tensors per level.
        for level_idx, pooled in enumerate(enc_features):
            all_features[f"level_{level_idx}"].append(pooled.squeeze(0).cpu().float().numpy())

        # Record the content indices per level.
        #  - Tuple masks: training-style n_views=2 with per-view masks (unused
        #    here since we run n_views=1, but handled for completeness).
        #  - Single-tensor mask: single-view forward.  When per-view learned
        #    masks are active the returned tensor is the view-specific mask
        #    (channel_logits_v1 when view_idx=1), so T2 must be stored in
        #    all_content_indices_v1 to preserve the per-view split.
        for lvl, mask in soft_masks.items():
            if isinstance(mask, tuple):
                mask_v0, mask_v1 = mask
                if modality == "T1" and lvl not in all_content_indices:
                    all_content_indices[lvl] = torch.where(mask_v0.bool())[-1].tolist()
                elif modality == "T2" and lvl not in all_content_indices_v1:
                    all_content_indices_v1[lvl] = torch.where(mask_v1.bool())[-1].tolist()
            else:
                if modality == "T1" and lvl not in all_content_indices:
                    all_content_indices[lvl] = torch.where(mask.bool())[-1].tolist()
                elif modality == "T2" and _has_per_view and lvl not in all_content_indices_v1:
                    all_content_indices_v1[lvl] = torch.where(mask.bool())[-1].tolist()

        all_labels.append(item["label"])
        all_modalities.append(modality)
        all_subjects.append(item.get("subject", idx))

# Convert to numpy
for key in all_features:
    all_features[key] = np.array(all_features[key])
all_labels = np.array(all_labels)
all_modalities = np.array(all_modalities)

print(f"\nDone. {len(all_labels)} samples total.")
for key, val in all_features.items():
    print(f"  {key}: {val.shape}")

# Derive content / style subsets from the hidden_channels features for each masked level.
all_style_indices = {}
all_style_indices_v1 = {}

for lvl in all_content_indices.keys():
    all_style_indices[lvl] = [i for i in range(hidden_channels) if i not in set(all_content_indices[lvl])]
    if _has_per_view and lvl in all_content_indices_v1:
        all_style_indices_v1[lvl] = [i for i in range(hidden_channels) if i not in set(all_content_indices_v1[lvl])]
    else:
        all_style_indices_v1[lvl] = all_style_indices[lvl]

if len(all_content_indices.get(0, [])) > 0:
    for lvl in all_content_indices.keys():
        print(f"\n--- Level {lvl} ---")
        print(f"  content indices v0 ({len(all_content_indices[lvl])} ch): {all_content_indices[lvl][:8]}...")
        print(f"  style indices   v0 ({len(all_style_indices[lvl])} ch):   {all_style_indices[lvl][:8]}...")
        if _has_per_view and lvl in all_content_indices_v1:
            print(f"  content indices v1 ({len(all_content_indices_v1[lvl])} ch): {all_content_indices_v1[lvl][:8]}...")
            print(f"  style indices   v1 ({len(all_style_indices_v1[lvl])} ch):   {all_style_indices_v1[lvl][:8]}...")

## 11. Codebook Analysis — Extract Indices

Run a separate forward pass with `return_recon=True` so that non-coarsest codebook
levels get proper decoder conditioning (without it they receive zero-padded input
and produce incorrect indices).

We collect:
- **Per-level codebook usage histograms** (normalized frequency of each entry per subject)
- **Raw codebook indices** for the code-replacement experiment later

In [ ]:
nb_entries = vqvae_module.codebooks[0].n_embed
print(f"Codebook: {nb_levels} levels, {nb_entries} entries each\n")

# Storage: index 0 = finest (level 0), index nb_levels-1 = coarsest
cb_histograms = [[] for _ in range(nb_levels)]  # per-subject normalized histograms
cb_labels = []  # diagnosis label per sample
cb_modalities = []  # "T1" or "T2"

# Per-class examples for code-replacement comparison (CN vs AD)
class_examples = {}

# Position-wise code frequency accumulators (T1 only).
# pos_counts[level] has shape (D_l, H_l, W_l, nb_entries) — lazily initialized.
pos_counts = [None for _ in range(nb_levels)]
n_t1_subjects = 0  # number of T1 subjects contributing to pos_counts

print(f"Extracting codebook indices from {len(items)} subjects (T1 + T2) ...")
print(f"Will keep one T1 example per class for code-replacement demo: {list(label_names.values())}")

for idx, item in enumerate(items):
    if idx % 20 == 0:
        print(f"  {idx}/{len(items)} ...")

    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for modality, key in [("T1", "image_t1"), ("T2", "image_t2")]:
        img = transformed[key].unsqueeze(0).to(DEVICE)
        _vi = 0 if modality == "T1" else 1

        with torch.no_grad():
            recon, diffs, _, _, _, id_outputs_raw, _, _ = vqvae_model(
                img,
                return_recon=True,
                pool_only=False,
                view_idx=_vi,
            )

        # id_outputs_raw is coarsest-first; reverse to finest-first
        id_outputs = id_outputs_raw[::-1]

        for lvl in range(nb_levels):
            hist = torch.bincount(id_outputs[lvl].reshape(-1), minlength=nb_entries)
            hist = hist.float() / hist.sum()
            cb_histograms[lvl].append(hist.cpu().numpy())

        cb_labels.append(item["label"])
        cb_modalities.append(modality)

        # Accumulate position-wise code counts (T1 only to avoid modality confound)
        if modality == "T1":
            n_t1_subjects += 1
            for lvl in range(nb_levels):
                ids_lvl = id_outputs[lvl].squeeze(0).cpu()  # (D_l, H_l, W_l)
                if pos_counts[lvl] is None:
                    spatial = ids_lvl.shape
                    pos_counts[lvl] = torch.zeros(*spatial, nb_entries, dtype=torch.long)
                # Scatter-add: for each position, increment the count for the observed code
                pos_counts[lvl].scatter_(-1, ids_lvl.unsqueeze(-1), 1)

        # Keep one T1 example per diagnosis class
        if modality == "T1":
            class_name = label_names[item["label"]]
            if class_name not in class_examples:
                t1_tensor = transformed[key]
                ex_affine = None
                if hasattr(t1_tensor, "affine"):
                    ex_affine = t1_tensor.affine.numpy()
                elif hasattr(t1_tensor, "meta") and "affine" in t1_tensor.meta:
                    ex_affine = t1_tensor.meta["affine"].numpy()

                class_examples[class_name] = {
                    "id_outputs": [id_outputs[lvl].clone() for lvl in range(nb_levels)],
                    "image": img.clone(),
                    "item": item,
                    "affine": ex_affine,
                }
                print(f"  ✓ Stored {class_name} example: subject {item.get('subject', idx)}")

# Stack
for lvl in range(nb_levels):
    cb_histograms[lvl] = np.array(cb_histograms[lvl])
cb_labels = np.array(cb_labels)
cb_modalities = np.array(cb_modalities)

# Backward compat
_fallback_class = list(class_examples.keys())[-1] if class_examples else None
last_id_outputs = class_examples[_fallback_class]["id_outputs"] if _fallback_class else None
last_image = class_examples[_fallback_class]["image"] if _fallback_class else None
last_item = class_examples[_fallback_class]["item"] if _fallback_class else None
last_affine = class_examples[_fallback_class]["affine"] if _fallback_class else None

print(f"\n✓ Codebook extraction complete ({cb_histograms[0].shape[0]} samples)")
print(f"  T1 subjects for position-wise analysis: {n_t1_subjects}")
for lvl in range(nb_levels):
    used = (cb_histograms[lvl].sum(axis=0) > 0).sum()
    sp = tuple(pos_counts[lvl].shape[:-1]) if pos_counts[lvl] is not None else "?"
    print(
        f"  Level {lvl}: histogram shape {cb_histograms[lvl].shape}, "
        f"codes used: {used}/{nb_entries}, code map spatial: {sp}"
    )
print(f"  Class examples stored: {list(class_examples.keys())}")

## 12-pre. Style Codebook Perplexity (capacity diagnostic)

If modality invariance fails at a given level while levels above succeed, one hypothesis is that the **style codebook at that level is capacity-starved** — it cannot represent the modality contrast, so modality info leaks back into content. This cell measures, per level:

- **Codebook size** `K`
- **Unique codes used** across all spatial positions / subjects
- **Perplexity** `exp(-Σ p_k log p_k)` of the marginal usage distribution (effective number of codes in use)
- **Utilization** = perplexity / K ∈ [0, 1]

Split by modality (T1 vs T2): if style is doing its job, T1 and T2 should concentrate on different codes (low Jensen–Shannon overlap). If the codebook is saturated (utilization near 1) **and** the per-level modality probe on style features is already high, the level is capacity-starved → **problem 3** (increase `--style-nb-entries` / style channels). If utilization is low / codebook is barely used, the issue is routing/pressure, not capacity → **problem 1**.


In [ ]:
# ── Style codebook perplexity per level ───────────────────────────
# Requires quantize_style=True. Reuses `items` / `val_transforms` / `DEVICE`
# from the content-codebook extraction cell above.

if not getattr(vqvae_module, "quantize_style", False):
    print("⚠ quantize_style is False — no style codebooks to analyze. Skipping.")
else:
    style_levels = sorted(vqvae_module.style_codebooks.keys(), key=int)
    style_sizes = {int(l): vqvae_module.style_nb_entries_per_level[int(l)] for l in style_levels}
    print(f"Style codebooks: levels {[int(l) for l in style_levels]}")
    for l in style_levels:
        print(f"  level {l}: K = {style_sizes[int(l)]}")
    print()

    # Accumulate counts per (level, modality)
    style_counts = {int(l): {"T1": None, "T2": None} for l in style_levels}

    def _to_counts(ids_flat, K):
        return torch.bincount(ids_flat, minlength=K).cpu().numpy().astype(np.int64)

    print(f"Running forward passes on {len(items)} subjects (T1 + T2) ...")
    for idx, item in enumerate(items):
        if idx % 20 == 0:
            print(f"  {idx}/{len(items)} ...")
        data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
        transformed = val_transforms(data_dict)
        for modality, key in [("T1", "image_t1"), ("T2", "image_t2")]:
            img = transformed[key].unsqueeze(0).to(DEVICE)
            _vi = 0 if modality == "T1" else 1
            with torch.no_grad():
                out = vqvae_model(img, return_recon=True, pool_only=False, view_idx=_vi)
            # 8-tuple: (..., style_id_outputs)
            style_ids = out[7]  # dict {lvl: LongTensor indices}
            for lvl, ids in style_ids.items():
                K = style_sizes[lvl]
                c = _to_counts(ids.reshape(-1), K)
                cur = style_counts[lvl][modality]
                style_counts[lvl][modality] = c if cur is None else cur + c

    # ── Perplexity / utilization per level ────────────────────────
    def _perplexity(counts):
        total = counts.sum()
        if total == 0:
            return 0.0, 0, np.zeros_like(counts, dtype=np.float64)
        p = counts.astype(np.float64) / total
        nz = p > 0
        H = -(p[nz] * np.log(p[nz])).sum()
        return float(np.exp(H)), int(nz.sum()), p

    def _jsd(p, q):
        m = 0.5 * (p + q)

        def _kl(a, b):
            mask = (a > 0) & (b > 0)
            return float((a[mask] * np.log(a[mask] / b[mask])).sum())

        return 0.5 * _kl(p, m) + 0.5 * _kl(q, m)

    print()
    print("=" * 88)
    print(f"{'Level':<6} {'K':>5} {'modality':>9} {'used':>6} {'perplex':>9} {'util':>7}  {'JSD(T1,T2)':>12}")
    print("-" * 88)
    for lvl in sorted(style_counts.keys()):
        K = style_sizes[lvl]
        c_t1 = style_counts[lvl]["T1"]
        c_t2 = style_counts[lvl]["T2"]
        c_all = (c_t1 if c_t1 is not None else 0) + (c_t2 if c_t2 is not None else 0)

        ppl_all, used_all, p_all = _perplexity(c_all)
        ppl_t1, used_t1, p_t1 = _perplexity(c_t1) if c_t1 is not None else (0.0, 0, None)
        ppl_t2, used_t2, p_t2 = _perplexity(c_t2) if c_t2 is not None else (0.0, 0, None)

        jsd = _jsd(p_t1, p_t2) if (p_t1 is not None and p_t2 is not None) else float("nan")

        print(f"{lvl:<6} {K:>5} {'all':>9} {used_all:>6} {ppl_all:>9.2f} {ppl_all/K:>7.2%}  {jsd:>12.4f}")
        print(f"{'':<6} {'':>5} {'T1':>9} {used_t1:>6} {ppl_t1:>9.2f} {ppl_t1/K:>7.2%}")
        print(f"{'':<6} {'':>5} {'T2':>9} {used_t2:>6} {ppl_t2:>9.2f} {ppl_t2/K:>7.2%}")
    print("=" * 88)
    print()
    print("Interpretation:")
    print("  • util → 1.0 and JSD(T1,T2) small  ⇒ codebook is saturated but not modality-specific:")
    print("                                         capacity-starved (problem 3).")
    print("  • util → 1.0 and JSD(T1,T2) large  ⇒ style is doing its job; look elsewhere.")
    print("  • util low                          ⇒ codebook under-used → not capacity; check")
    print("                                         recon/contrastive gradient balance (problem 1).")

    # ── Plot marginal usage histograms per level, overlaid by modality ──────
    n_lvls = len(style_counts)
    fig, axes = plt.subplots(1, n_lvls, figsize=(5 * n_lvls, 3.5), squeeze=False)
    for i, lvl in enumerate(sorted(style_counts.keys())):
        K = style_sizes[lvl]
        ax = axes[0, i]
        c_t1 = style_counts[lvl]["T1"]
        c_t2 = style_counts[lvl]["T2"]
        order = np.argsort(-(c_t1 + c_t2))  # sort by combined frequency
        x = np.arange(K)
        if c_t1 is not None:
            ax.bar(x - 0.2, c_t1[order] / c_t1.sum(), width=0.4, label="T1", alpha=0.8)
        if c_t2 is not None:
            ax.bar(x + 0.2, c_t2[order] / c_t2.sum(), width=0.4, label="T2", alpha=0.8)
        ax.set_title(f"Level {lvl} style codebook (K={K})")
        ax.set_xlabel("code index (sorted by total usage)")
        ax.set_ylabel("P(code)")
        ax.legend()
    plt.tight_layout()
    plt.show()

## 12-pre-b. Style-Path Utilisation Diagnostics

Three complementary checks for whether the style path is actually being used at level 0. If it isn't, modality information must flow through **content**, which explains poor L0 invariance even when style has ample (continuous) capacity.

1. **Activation norms.** Compare ‖style_L0‖ / ‖content_L0‖ over a batch of subjects. If style norm is much smaller than content norm (or near zero), style is going dead.
2. **Recon ablation.** Re-run the model with `decoders[0]`'s style input zeroed. If reconstruction barely changes, the decoder isn't conditioning on style → modality has to be in content.
3. **Decoder-0 style-path weight norm.** Inspect the weights in `decoders[0]` that consume style (FiLM conv, or the style slice of `final_conv`). Near-zero weights → decoder has learned to ignore style.


In [ ]:
# ── Diagnostic 1: Activation norms of content vs style at L0 ─────────────────
import numpy as np

content_idx = np.array(all_content_indices[0]) if 0 in all_content_indices else np.arange(hidden_channels)
style_idx_arr = np.array(all_style_indices.get(0, [])) if len(all_style_indices.get(0, [])) > 0 else None
if style_idx_arr is None or len(style_idx_arr) == 0:
    print("⚠ No style channels at level 0 — skipping style-path diagnostics.")
else:
    enc_stack = vqvae_module.encoders

    norms_content = {"T1": [], "T2": []}
    norms_style = {"T1": [], "T2": []}
    per_ch_style_means = {"T1": [], "T2": []}

    print(f"Collecting L0 activation norms over {len(items)} subjects (T1 + T2) ...")
    for i, item in enumerate(items):
        if i % 20 == 0:
            print(f"  {i}/{len(items)} ...")
        data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
        transformed = val_transforms(data_dict)
        for mod, key in [("T1", "image_t1"), ("T2", "image_t2")]:
            img = transformed[key].unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                enc_out = enc_stack[0](img)  # (1, hidden_channels, D, H, W)
            feat = enc_out.squeeze(0).cpu().numpy()
            c_feat = feat[content_idx]
            s_feat = feat[style_idx_arr]
            # Per-element RMS so channel counts don't bias comparison
            norms_content[mod].append(float(np.sqrt((c_feat**2).mean())))
            norms_style[mod].append(float(np.sqrt((s_feat**2).mean())))
            per_ch_style_means[mod].append(np.abs(s_feat).mean(axis=(1, 2, 3)))  # (S,)

    print()
    print("=" * 72)
    print(f"{'':<10} {'content RMS':>14} {'style RMS':>14} {'style/content':>16}")
    print("-" * 72)
    for mod in ["T1", "T2"]:
        cm = np.mean(norms_content[mod])
        sm = np.mean(norms_style[mod])
        print(f"{mod:<10} {cm:>14.4f} {sm:>14.4f} {sm/cm:>16.4f}")
    print("=" * 72)
    print()
    print("Rule of thumb: ratio < ~0.1 suggests style is under-used; < ~0.01 suggests collapse.")

    # Per-channel style activation magnitude — detects partial/selective collapse
    ch_mean_t1 = np.mean(np.stack(per_ch_style_means["T1"]), axis=0)
    ch_mean_t2 = np.mean(np.stack(per_ch_style_means["T2"]), axis=0)
    fig, ax = plt.subplots(1, 1, figsize=(max(6, 0.3 * len(style_idx_arr)), 3.5))
    x = np.arange(len(style_idx_arr))
    ax.bar(x - 0.2, ch_mean_t1, width=0.4, label="T1", alpha=0.8)
    ax.bar(x + 0.2, ch_mean_t2, width=0.4, label="T2", alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in style_idx_arr], rotation=90, fontsize=8)
    ax.set_xlabel("style channel index")
    ax.set_ylabel("mean |activation|")
    ax.set_title("L0 per-style-channel mean activation (T1 vs T2)")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Diagnostic 2: Recon ablation — zero out style into decoders[0] ──────────
# Compare recon MSE with normal style vs with style forced to zero at L0.
# If MSE barely moves, the decoder isn't using L0 style → modality must travel
# through content to explain T1 vs T2 reconstructions.

import torch.nn.functional as F
from contextlib import contextmanager


@contextmanager
def zero_style_in_decoder0():
    """Temporarily replace decoders[0].forward so style is zeroed."""
    dec0 = vqvae_module.decoders[0]
    orig_forward = dec0.forward

    def patched(x, style=None):
        if style is not None:
            style = torch.zeros_like(style)
        return orig_forward(x, style=style)

    dec0.forward = patched
    try:
        yield
    finally:
        dec0.forward = orig_forward


mse_normal = {"T1": [], "T2": []}
mse_ablated = {"T1": [], "T2": []}

# Subsample for speed — full recon passes are more expensive than encoder-only
N_ABLATION = min(40, len(items))
print(f"Running recon ablation on {N_ABLATION} subjects (T1 + T2) ...")
for i, item in enumerate(items[:N_ABLATION]):
    if i % 10 == 0:
        print(f"  {i}/{N_ABLATION} ...")
    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)
    for mod, key in [("T1", "image_t1"), ("T2", "image_t2")]:
        img = transformed[key].unsqueeze(0).to(DEVICE)
        _vi = 0 if mod == "T1" else 1
        with torch.no_grad():
            recon_n = vqvae_model(img, return_recon=True, pool_only=False, view_idx=_vi)[0]
            with zero_style_in_decoder0():
                recon_a = vqvae_model(img, return_recon=True, pool_only=False, view_idx=_vi)[0]
        if recon_n.shape != img.shape:
            recon_n = F.interpolate(recon_n, size=img.shape[2:], mode="trilinear", align_corners=False)
            recon_a = F.interpolate(recon_a, size=img.shape[2:], mode="trilinear", align_corners=False)
        mse_normal[mod].append(F.mse_loss(recon_n, img).item())
        mse_ablated[mod].append(F.mse_loss(recon_a, img).item())

print()
print("=" * 80)
print(f"{'modality':<10} {'MSE (normal)':>14} {'MSE (style=0)':>16} {'Δ abs':>12} {'Δ rel':>10}")
print("-" * 80)
for mod in ["T1", "T2"]:
    mn = float(np.mean(mse_normal[mod]))
    ma = float(np.mean(mse_ablated[mod]))
    delta = ma - mn
    rel = delta / mn if mn > 0 else float("nan")
    print(f"{mod:<10} {mn:>14.6f} {ma:>16.6f} {delta:>12.6f} {rel:>10.2%}")
print("=" * 80)
print()
print("Interpretation:")
print("  • Δ rel < ~5%    ⇒ decoder barely uses L0 style → style path is dead,")
print("                       modality must be encoded in content (problem 1 confirmed).")
print("  • Δ rel large  ⇒ decoder relies on L0 style; content-modality leakage")
print("                       is coming from another mechanism (mask routing, grad balance).")
print("  Also check: if T2 Δ ≫ T1 Δ, the style path is asymmetrically used across modalities.")

In [ ]:
# ── Diagnostic 3: Decoder-0 style-path weight norms ─────────────────────────
# Inspect how much weight the decoder places on its style input. Near-zero
# norms ⇒ decoder has learned to ignore style (posterior-collapse analogue).

dec0 = vqvae_module.decoders[0]
print(f"decoders[0].style_channels     : {dec0.style_channels}")
print(f"decoders[0].style_injection_mode: {dec0.style_injection_mode}")
print()

if dec0.style_channels == 0:
    print("No style channels on decoder 0 — nothing to inspect.")
elif dec0.film_layers is not None:
    # FiLM mode: each SpatialFiLM has a 1×1×1 conv (style_channels → 2 * feature_channels)
    print("FiLM mode — per-layer style-conv weight norms:")
    print(f"  {'layer':<10} {'conv shape':<22} {'weight L2':>12} {'mean|w|':>12} {'bias L2':>10}")
    print("  " + "-" * 72)
    all_w_norms = []
    for i, film in enumerate(dec0.film_layers):
        w = film.conv.weight.detach().cpu()
        b = film.conv.bias.detach().cpu() if film.conv.bias is not None else None
        wn = float(w.norm())
        wm = float(w.abs().mean())
        bn = float(b.norm()) if b is not None else 0.0
        all_w_norms.append(wn)
        print(f"  film_{i:<5} {str(tuple(w.shape)):<22} {wn:>12.4e} {wm:>12.4e} {bn:>10.4e}")
    print()
    print("Note: FiLM convs are initialised to zero so the decoder starts style-agnostic.")
    print("If all norms are still ~0 after training, style is being ignored.")
    if max(all_w_norms) < 1e-3:
        print("  ⚠ All FiLM weight norms are near zero — style is effectively unused.")
elif dec0.final_conv is not None:
    # Concat mode: style appears as the last `style_channels` input channels of final_conv[0]
    conv = dec0.final_conv[0]
    w = conv.weight.detach().cpu()  # (out, in, kD, kH, kW)
    total_in = w.shape[1]
    sc = dec0.style_channels
    content_slice = w[:, : total_in - sc]
    style_slice = w[:, total_in - sc :]
    content_norm = float(content_slice.norm())
    style_norm = float(style_slice.norm())
    content_mean = float(content_slice.abs().mean())
    style_mean = float(style_slice.abs().mean())
    print("Concat mode — final_conv input-channel weight norms:")
    print(f"  content-path weight L2 : {content_norm:.4e}   mean|w|: {content_mean:.4e}")
    print(f"  style-path   weight L2 : {style_norm:.4e}   mean|w|: {style_mean:.4e}")
    print(f"  style / content ratio  : {style_norm / content_norm:.4f}")
    print()
    if style_norm / content_norm < 0.05:
        print("  ⚠ Style-path weights are ≪ content-path — decoder has largely ignored style.")
    # Per-channel style weight norms — detects which style channels are being used
    per_ch = style_slice.reshape(style_slice.shape[0], sc, -1).pow(2).sum(dim=(0, 2)).sqrt().numpy()
    fig, ax = plt.subplots(1, 1, figsize=(max(6, 0.3 * sc), 3.2))
    x = np.arange(sc)
    ax.bar(x, per_ch)
    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in (style_idx_arr if style_idx_arr is not None else x)], rotation=90, fontsize=8)
    ax.set_xlabel("style channel (input to final_conv)")
    ax.set_ylabel("weight L2 across output ch + kernel")
    ax.set_title("decoders[0].final_conv per-style-channel weight norm")
    plt.tight_layout()
    plt.show()

## 12. Codebook Usage by Diagnosis Class

Mean codebook entry frequency per class, shown as overlaid histograms and a heatmap.
Entries that are heavily used by one class but not others indicate class-specific
structural patterns captured by that code vector.

In [ ]:
import seaborn as sns

# Use T1 only so modality doesn't confound the class comparison
t1_cb_mask = cb_modalities == "T1"
t1_cb_labels = cb_labels[t1_cb_mask]

for lvl in range(nb_levels):
    hists = cb_histograms[lvl][t1_cb_mask]

    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    fig.suptitle(f"Level {lvl} — Codebook Usage by Diagnosis (T1 only)", fontsize=14)

    # Mean usage histogram per class
    ax = axes[0]
    for lab_idx, lab_name in label_names.items():
        mask = t1_cb_labels == lab_idx
        mean_usage = hists[mask].mean(axis=0)
        ax.bar(np.arange(nb_entries), mean_usage, alpha=0.5, label=lab_name)
    ax.set_xlabel("Codebook entry")
    ax.set_ylabel("Mean frequency")
    ax.set_title("Mean codebook usage")
    ax.legend()

    # Heatmap: classes x entries
    ax = axes[1]
    usage_matrix = np.zeros((len(label_names), nb_entries))
    for lab_idx in label_names:
        usage_matrix[lab_idx] = hists[t1_cb_labels == lab_idx].mean(axis=0)
    sns.heatmap(
        usage_matrix,
        ax=ax,
        cmap="viridis",
        yticklabels=[label_names[i] for i in sorted(label_names)],
        xticklabels=False,
    )
    ax.set_xlabel("Codebook entry")
    ax.set_title("Usage heatmap (class x entry)")

    plt.tight_layout()
    plt.savefig(f"codebook_usage_level{lvl}.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Print top codes with biggest cross-class difference
    max_diff = usage_matrix.max(axis=0) - usage_matrix.min(axis=0)
    dominant_class = np.argmax(usage_matrix, axis=0)
    top_diff = np.argsort(max_diff)[::-1][:10]
    print(f"  Level {lvl} — codes with largest cross-class difference:")
    print(f"  {'Code':>6s}  {'Dominant':>8s}  {'MaxFreq':>8s}  {'MinFreq':>8s}  {'Diff':>8s}")
    for idx in top_diff:
        dom = label_names[dominant_class[idx]]
        print(
            f"  {idx:6d}  {dom:>8s}  {usage_matrix[:, idx].max():8.4f}  "
            f"{usage_matrix[:, idx].min():8.4f}  {max_diff[idx]:8.4f}"
        )
    print()

## 12a. Codebook Usage by Demographic Group & Diagnosis

Merge codebook histograms with demographic metadata from `merged_data.csv` and compare
codebook entry distributions across **gender**, **race/ethnicity**, **education**, and **diagnosis**.

In [ ]:
# ── Load demographic metadata and merge with codebook histograms ─────────────
demo_df = pd.read_csv(os.path.join(os.path.dirname(CSV_PATH), "merged_data.csv"))

# Build a lookup from Subject -> demographic info (keep first row per subject)
demo_lookup = demo_df.drop_duplicates(subset="Subject").set_index("Subject")

# The T1 entries in cb_histograms are at even indices (0, 2, 4, ...),
# one per item in `items`. Build a DataFrame aligned with the T1-filtered hists.
t1_subjects = [item["subject"] for item in items]
cb_demo = pd.DataFrame({"Subject": t1_subjects})
cb_demo = cb_demo.join(demo_lookup, on="Subject", rsuffix="_dup")

# Decode demographic columns
gender_map = {1.0: "Male", 2.0: "Female"}
race_map = {
    1: "Am Indian/Alaskan",
    2: "Asian",
    3: "Native Hawaiian/Pacific Isl",
    4: "Black",
    5: "White",
    6: "More than one",
    7: "Unknown",
}

cb_demo["Gender"] = cb_demo["PTGENDER"].map(gender_map)
# PTRACCAT can have multi-race codes like "1|5"; take first for simplicity
cb_demo["Race"] = cb_demo["PTRACCAT"].apply(
    lambda x: race_map.get(int(str(x).split("|")[0]), f"Code {x}") if pd.notna(x) else "Unknown"
)
cb_demo["Education"] = pd.cut(cb_demo["PTEDUCAT"], bins=[0, 12, 16, 25], labels=["≤12 yrs", "13-16 yrs", "17+ yrs"])

# Verify alignment: cb_demo should have exactly as many rows as T1 histograms
n_t1 = int(t1_cb_mask.sum())
assert len(cb_demo) == n_t1, f"cb_demo has {len(cb_demo)} rows but expected {n_t1} T1 samples"

print(f"Matched {cb_demo['Gender'].notna().sum()}/{len(cb_demo)} T1 samples to demographics")
print(f"\nGroup counts:")
for col in ["Group", "Gender", "Race", "Education"]:
    print(f"  {col}: {cb_demo[col].value_counts().to_dict()}")

In [ ]:
# ── Plot codebook usage differences across demographic groups & diagnosis ─────
from scipy.stats import entropy as kl_div

grouping_cols = {
    "Group": "Diagnosis",
    "Gender": "Gender",
    "Race": "Race/Ethnicity",
    "Education": "Education Level",
}

t1_hists = {lvl: cb_histograms[lvl][t1_cb_mask] for lvl in range(nb_levels)}

for lvl in range(nb_levels):
    hists = t1_hists[lvl]

    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    fig.suptitle(
        f"Level {lvl} — Codebook Entry Distribution by Demographic Group (T1 only)", fontsize=15, fontweight="bold"
    )

    for ax, (col, title) in zip(axes.flat, grouping_cols.items()):
        groups = cb_demo[col].dropna().unique()
        groups = sorted(groups, key=str)

        # Compute mean histogram per group
        mean_hists = {}
        for g in groups:
            mask = (cb_demo[col] == g).values
            if mask.sum() < 2:
                continue
            mean_hists[g] = hists[mask].mean(axis=0)

        # Plot as overlaid bar chart
        x = np.arange(nb_entries)
        n_groups = len(mean_hists)
        width = 0.8 / max(n_groups, 1)
        for i, (g, mh) in enumerate(mean_hists.items()):
            offset = (i - n_groups / 2 + 0.5) * width
            ax.bar(x + offset, mh, width=width, alpha=0.6, label=f"{g} (n={int((cb_demo[col]==g).sum())})")

        ax.set_xlabel("Codebook entry")
        ax.set_ylabel("Mean frequency")
        ax.set_title(f"By {title}")
        ax.legend(fontsize=8, loc="upper right")

    plt.tight_layout()
    plt.savefig(f"codebook_demographic_level{lvl}.png", dpi=150, bbox_inches="tight")
    plt.show()

    # ── Difference heatmaps: for each grouping, show per-entry deviation from overall mean ──
    fig, axes = plt.subplots(2, 2, figsize=(20, 10))
    fig.suptitle(f"Level {lvl} — Codebook Usage Deviation from Overall Mean", fontsize=15, fontweight="bold")

    overall_mean = hists.mean(axis=0)

    for ax, (col, title) in zip(axes.flat, grouping_cols.items()):
        groups = sorted(cb_demo[col].dropna().unique(), key=str)
        group_names = []
        diff_matrix = []
        for g in groups:
            mask = (cb_demo[col] == g).values
            if mask.sum() < 2:
                continue
            group_names.append(f"{g} (n={int(mask.sum())})")
            diff_matrix.append(hists[mask].mean(axis=0) - overall_mean)

        if len(diff_matrix) == 0:
            ax.set_visible(False)
            continue

        diff_matrix = np.array(diff_matrix)
        vmax = np.abs(diff_matrix).max()
        sns.heatmap(
            diff_matrix,
            ax=ax,
            cmap="RdBu_r",
            center=0,
            vmin=-vmax,
            vmax=vmax,
            yticklabels=group_names,
            xticklabels=False,
        )
        ax.set_xlabel("Codebook entry")
        ax.set_title(f"By {title}")

    plt.tight_layout()
    plt.savefig(f"codebook_demographic_diff_level{lvl}.png", dpi=150, bbox_inches="tight")
    plt.show()

    # ── Print KL divergence between group pairs ─────────────────────────────────
    print(f"\n  Level {lvl} — KL divergence between group-mean distributions:")
    for col, title in grouping_cols.items():
        groups = sorted(cb_demo[col].dropna().unique(), key=str)
        mean_h = {}
        for g in groups:
            mask = (cb_demo[col] == g).values
            if mask.sum() >= 2:
                h = hists[mask].mean(axis=0)
                mean_h[g] = h / h.sum()  # re-normalize
        gnames = list(mean_h.keys())
        if len(gnames) < 2:
            continue
        print(f"    {title}:")
        for i in range(len(gnames)):
            for j in range(i + 1, len(gnames)):
                kl = kl_div(mean_h[gnames[i]] + 1e-10, mean_h[gnames[j]] + 1e-10)
                print(f"      {gnames[i]} vs {gnames[j]}: KL = {kl:.6f}")
    print()

## 12b. Dominant Code Pairs by Spatial Position

For each spatial position in the code map, we look at which codebook entries appear
most often across all T1 subjects. If the **top-2 codes** at a position both exceed
a frequency threshold (e.g., 30% of subjects), that position exhibits a **dominant
pair** — meaning the model consistently alternates between two codes at that location.

We then aggregate these pairs across all positions and levels to find the most
common dominant pairs in the dataset. This reveals:
- Whether the codebook is being used efficiently (many distinct pairs) or
  degenerately (a few codes dominate everywhere)
- Spatial structure in code assignment (e.g., do certain brain regions always
  alternate between the same two codes?)

In [ ]:
from collections import Counter

# ── Configuration ────────────────────────────────────────────────
FREQ_THRESHOLD = 0.30  # both top-2 codes must appear in ≥ this fraction of subjects
TOP_K_PAIRS = 20  # how many top pairs to show per level

print(
    f"Frequency threshold: both codes must appear in ≥ {FREQ_THRESHOLD:.0%} of "
    f"{n_t1_subjects} T1 subjects (≥ {int(FREQ_THRESHOLD * n_t1_subjects)} subjects)\n"
)

all_level_results = {}

for lvl in range(nb_levels):
    counts = pos_counts[lvl]  # (D, H, W, nb_entries), long
    spatial_shape = counts.shape[:-1]
    n_positions = counts[..., 0].numel()

    # Convert to frequencies
    freqs = counts.float() / n_t1_subjects  # (D, H, W, nb_entries)

    # Top-2 codes and their frequencies at each position
    top2_freqs, top2_codes = freqs.topk(2, dim=-1)  # each (D, H, W, 2)
    freq1 = top2_freqs[..., 0]  # most common
    freq2 = top2_freqs[..., 1]  # second most common
    code1 = top2_codes[..., 0]
    code2 = top2_codes[..., 1]

    # Positions where both top-2 codes exceed threshold
    dominant_mask = (freq1 >= FREQ_THRESHOLD) & (freq2 >= FREQ_THRESHOLD)
    n_dominant = dominant_mask.sum().item()

    print(f"{'='*65}")
    print(f"Level {lvl}  —  spatial {tuple(spatial_shape)}  " f"({n_positions} positions)")
    print(
        f"  Positions with dominant pair (both ≥ {FREQ_THRESHOLD:.0%}): "
        f"{n_dominant} / {n_positions} ({n_dominant/n_positions:.1%})"
    )

    if n_dominant == 0:
        print("  No dominant pairs found at this level. " "Try lowering FREQ_THRESHOLD.\n")
        all_level_results[lvl] = {"n_dominant": 0, "pair_counts": Counter()}
        continue

    # Extract the dominant pairs as sorted tuples (so (a,b) == (b,a))
    c1 = code1[dominant_mask].numpy()
    c2 = code2[dominant_mask].numpy()
    f1 = freq1[dominant_mask].numpy()
    f2 = freq2[dominant_mask].numpy()

    pair_counter = Counter()
    pair_freq_sum = {}  # accumulate frequencies for averaging
    for i in range(len(c1)):
        pair = tuple(sorted([int(c1[i]), int(c2[i])]))
        pair_counter[pair] += 1
        if pair not in pair_freq_sum:
            pair_freq_sum[pair] = [0.0, 0.0, 0]  # sum_freq1, sum_freq2, count
        pair_freq_sum[pair][0] += f1[i]
        pair_freq_sum[pair][1] += f2[i]
        pair_freq_sum[pair][2] += 1

    all_level_results[lvl] = {
        "n_dominant": n_dominant,
        "pair_counts": pair_counter,
        "dominant_mask": dominant_mask,
    }

    # Print top pairs
    print(f"\n  Top-{min(TOP_K_PAIRS, len(pair_counter))} dominant pairs:")
    print(f"  {'Pair':<16} {'# positions':>12} {'% of dominant':>14} " f"{'Avg freq code1':>15} {'Avg freq code2':>15}")
    print(f"  {'-'*74}")

    for pair, count in pair_counter.most_common(TOP_K_PAIRS):
        pct = count / n_dominant * 100
        s = pair_freq_sum[pair]
        avg_f1 = s[0] / s[2]
        avg_f2 = s[1] / s[2]
        print(
            f"  ({pair[0]:>3d}, {pair[1]:>3d})     {count:>8d}     {pct:>10.1f}%"
            f"     {avg_f1:>11.1%}     {avg_f2:>11.1%}"
        )

    # Summary stats
    n_unique_pairs = len(pair_counter)
    n_unique_codes = len(set(c for pair in pair_counter for c in pair))
    print(f"\n  Unique dominant pairs: {n_unique_pairs}")
    print(f"  Unique codes involved: {n_unique_codes} / {nb_entries}")

    # Distribution of how many positions each pair covers
    pair_sizes = list(pair_counter.values())
    print(
        f"  Positions per pair: min={min(pair_sizes)}, "
        f"median={sorted(pair_sizes)[len(pair_sizes)//2]}, "
        f"max={max(pair_sizes)}"
    )
    print()

# ── Cross-level summary ──────────────────────────────────────────
print("=" * 65)
print("Cross-level summary")
print("=" * 65)

# Find pairs that appear as dominant at multiple levels
all_pairs = Counter()
for lvl in range(nb_levels):
    for pair in all_level_results[lvl]["pair_counts"]:
        all_pairs[pair] += all_level_results[lvl]["pair_counts"][pair]

# Pairs that are dominant at more than one level
multi_level_pairs = []
for pair in all_pairs:
    levels_present = [lvl for lvl in range(nb_levels) if pair in all_level_results[lvl]["pair_counts"]]
    if len(levels_present) > 1:
        total = sum(all_level_results[l]["pair_counts"][pair] for l in levels_present)
        multi_level_pairs.append((pair, levels_present, total))

multi_level_pairs.sort(key=lambda x: -x[2])

if multi_level_pairs:
    print(f"\nPairs dominant at multiple levels ({len(multi_level_pairs)} pairs):")
    for pair, levels, total in multi_level_pairs[:15]:
        lvl_str = ", ".join(str(l) for l in levels)
        print(f"  ({pair[0]:>3d}, {pair[1]:>3d})  levels=[{lvl_str}]  " f"total positions={total}")
else:
    print("\nNo pairs are dominant at multiple levels.")

# Overall code utilization in dominant positions
all_dominant_codes = set()
for lvl in range(nb_levels):
    for pair in all_level_results[lvl]["pair_counts"]:
        all_dominant_codes.update(pair)
print(
    f"\nTotal unique codes in dominant pairs (all levels): "
    f"{len(all_dominant_codes)} / {nb_entries} "
    f"({len(all_dominant_codes)/nb_entries:.1%})"
)

## 13. Most Discriminative Codes — Mutual Information & Chi-squared

**Mutual information** measures how much knowing a codebook entry's usage reduces
uncertainty about the diagnosis label (higher = more informative).

**Chi-squared** tests whether the usage distribution of each entry is independent
of the class label (higher statistic = stronger association).

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import chi2_contingency

for lvl in range(nb_levels):
    hists = cb_histograms[lvl][t1_cb_mask]
    y = t1_cb_labels

    # ── Mutual Information ────────────────────────────────────────────
    mi_scores = mutual_info_classif(hists, y, discrete_features=False, random_state=42)

    fig, axes = plt.subplots(1, 2, figsize=(18, 4))
    fig.suptitle(f"Level {lvl} — Codebook Discriminativeness (T1 only)", fontsize=14)

    ax = axes[0]
    ax.bar(np.arange(nb_entries), mi_scores, color="darkorange", width=1.0)
    ax.set_xlabel("Codebook entry")
    ax.set_ylabel("Mutual information (nats)")
    ax.set_title("MI: codebook entry usage vs diagnosis")

    top_mi = np.argsort(mi_scores)[::-1][:10]
    print(f"Level {lvl} — top-10 MI codes: {top_mi.tolist()}")
    print(f"  MI values: {mi_scores[top_mi].round(4).tolist()}")

    # ── Chi-squared ───────────────────────────────────────────────────
    chi2_scores = np.zeros(nb_entries)
    for code_idx in range(nb_entries):
        usage = hists[:, code_idx]
        # Bin non-zero usage into terciles for the contingency table
        nonzero = usage[usage > 0]
        if len(nonzero) < 10:
            continue
        bins = np.quantile(nonzero, [0.33, 0.66])
        if len(np.unique(bins)) < 2:
            continue
        digitized = np.digitize(usage, bins)
        contingency = pd.crosstab(digitized, y)
        if contingency.shape[0] > 1 and contingency.shape[1] > 1:
            chi2, p, _, _ = chi2_contingency(contingency)
            chi2_scores[code_idx] = chi2

    TOP_K = 20
    top_chi2 = np.argsort(chi2_scores)[::-1][:TOP_K]

    ax = axes[1]
    ax.bar(range(TOP_K), chi2_scores[top_chi2], color="steelblue")
    ax.set_xticks(range(TOP_K))
    ax.set_xticklabels(top_chi2, rotation=45)
    ax.set_xlabel("Codebook entry index")
    ax.set_ylabel("Chi-squared statistic")
    ax.set_title(f"Top {TOP_K} most class-discriminative codes (chi2)")

    plt.tight_layout()
    plt.savefig(f"codebook_discriminativeness_level{lvl}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print()

## 14. PCA & t-SNE of Codebook Usage Histograms

Each subject is represented by a vector of codebook entry frequencies. If the
codebook has learned class-relevant structure, subjects with the same diagnosis
should cluster together in this histogram space.

In [ ]:
for lvl in range(nb_levels):
    hists = cb_histograms[lvl][t1_cb_mask]

    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(hists)
    ev = pca.explained_variance_ratio_

    tsne = TSNE(n_components=2, perplexity=min(30, len(hists) - 1), random_state=42)
    X_tsne = tsne.fit_transform(hists)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Level {lvl} — Codebook Histogram Embeddings (T1 only)", fontsize=14)

    for lab_idx, lab_name in label_names.items():
        m = t1_cb_labels == lab_idx
        axes[0].scatter(X_pca[m, 0], X_pca[m, 1], c=[colors_by_label[lab_idx]], label=lab_name, alpha=0.6, s=18)
        axes[1].scatter(X_tsne[m, 0], X_tsne[m, 1], c=[colors_by_label[lab_idx]], label=lab_name, alpha=0.6, s=18)

    axes[0].set_title(f"PCA ({ev.sum()*100:.1f}% var)")
    axes[0].set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
    axes[0].set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")
    axes[0].legend(fontsize=8)

    axes[1].set_title("t-SNE")
    axes[1].set_xlabel("t-SNE 1")
    axes[1].set_ylabel("t-SNE 2")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(f"codebook_histogram_embeddings_level{lvl}.png", dpi=150, bbox_inches="tight")
    plt.show()

# Combined: concatenate histograms across all levels
X_all = np.concatenate([cb_histograms[lvl][t1_cb_mask] for lvl in range(nb_levels)], axis=1)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_all)
ev = pca.explained_variance_ratio_
tsne = TSNE(n_components=2, perplexity=min(30, len(X_all) - 1), random_state=42)
X_tsne = tsne.fit_transform(X_all)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("All Levels Combined — Codebook Histogram Embeddings (T1 only)", fontsize=14)
for lab_idx, lab_name in label_names.items():
    m = t1_cb_labels == lab_idx
    axes[0].scatter(X_pca[m, 0], X_pca[m, 1], c=[colors_by_label[lab_idx]], label=lab_name, alpha=0.6, s=18)
    axes[1].scatter(X_tsne[m, 0], X_tsne[m, 1], c=[colors_by_label[lab_idx]], label=lab_name, alpha=0.6, s=18)
axes[0].set_title(f"PCA ({ev.sum()*100:.1f}% var)")
axes[0].legend(fontsize=8)
axes[1].set_title("t-SNE")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig("codebook_histogram_embeddings_all_levels.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved codebook histogram embedding plots")